# EDA final — vitais sintéticos (Limen Épico 8 / E8.2)

> Protótipo acadêmico — **não** é um dispositivo médico.

Complementa [`eda_vitals_inicial.ipynb`](eda_vitals_inicial.ipynb): resume o
catálogo de **referência metodológica**, inspeciona fixtures versionadas e
documenta faixas usadas na calibração.

**Runtime/CI não baixam** Kaggle, HuggingFace nem PhysioNet — usam só
`data/fixtures/vitals/`.

## Catálogo (referência)

| Papel | Dataset | URL |
| ----- | ------- | --- |
| Primário | Human vital signs (Kaggle) | https://www.kaggle.com/datasets/engrarri21/human-vital-signs |
| Eventos (opc.) | Patient Vital Signs and Event Tracking | https://www.kaggle.com/datasets/parmajha/patient-vital-signs-and-event-tracking |
| Deterioração | Hospital Deterioration | https://www.kaggle.com/datasets/tarekmasryo/hospital-deterioration-dataset |
| PhysioNet (PDF/metodologia) | Challenge 2019 / MC-MED | https://physionet.org/content/challenge-2019/ · https://physionet.org/content/mc-med/1.0.0/ |

Brutos grandes: `data/raw/` (gitignored). Regenerar fixtures:
`python scripts/calibrate_vitals.py`.

In [ ]:
from pathlib import Path
import csv
import statistics

ROOT = Path("..").resolve()
VITALS = ROOT / "data" / "fixtures" / "vitals"
files = sorted(VITALS.glob("vitals_*.csv"))
assert files, "Fixtures ausentes — rode scripts/calibrate_vitals.py"
print("Fixtures:", [p.name for p in files])

summary = []
for path in files:
    with path.open(newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
    hrs = [float(r["heart_rate"]) for r in rows]
    spo2 = [float(r["spo2"]) for r in rows]
    anomalies = sum(1 for r in rows if r["label"] == "anomaly")
    summary.append(
        {
            "file": path.name,
            "n": len(rows),
            "hr_mean": round(statistics.fmean(hrs), 2),
            "hr_min": min(hrs),
            "hr_max": max(hrs),
            "spo2_min": min(spo2),
            "spo2_max": max(spo2),
            "anomalies": anomalies,
        }
    )
    print(
        f"{path.name}: n={len(rows)} HR μ={statistics.fmean(hrs):.1f} "
        f"[{min(hrs):.1f},{max(hrs):.1f}] "
        f"SpO2[{min(spo2):.1f},{max(spo2):.1f}] anomalies={anomalies}"
    )
summary

## Faixas e calibração (fixtures → Risco)

| Fixture | Intenção de Risco | Anomalia injetada |
| ------- | ----------------- | ----------------- |
| `vitals_normal.csv` | BAIXO | nenhuma |
| `vitals_medium.csv` | MEDIO | a partir de `t≈1200` s |
| `vitals_high.csv` | ALTO | a partir de `t≈900` s |

- Faixas basais: adulto em repouso (literatura + CSVs Kaggle de referência).
- PhysioNet Challenge 2019 / MC-MED: **referência metodológica** no relatório;
  **não** entram no runtime nem neste notebook como download.
- Limiares BAIXO/MEDIO/ALTO: spec
  `specs/epic-03-caso-fila/03-caso-vitais-risco.md` + ADR 0008.
- Sem PHI: sem CPF, nome, prontuário ou IDs reais.

README das fixtures:
[`data/fixtures/vitals/README.md`](../data/fixtures/vitals/README.md).